# 16. YOLO 영상 실습

이 노트북은 `15_YOLO_이미지_실습.ipynb`의 이미지 추론 흐름을 영상 프레임 단위로 확장합니다.

영상 탐지는 특별한 다른 문제가 아니라, 이미지를 여러 장 순서대로 처리하는 문제로 볼 수 있습니다. 다만 프레임 수가 많아지면서 속도, 저장, 시각화, threshold 관리가 중요해집니다.

이번 노트북의 목표는 다음과 같습니다.

- 영상 탐지를 프레임 반복 처리 관점에서 이해합니다.
- 예제 프레임 시퀀스에 탐지 결과를 그립니다.
- 실제 YOLO 모델과 OpenCV 영상 입력을 연결하는 코드를 확인합니다.
- FPS와 frame stride가 결과와 속도에 주는 영향을 이해합니다.

In [ ]:
from pathlib import Path
import time

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.unicode_minus'] = False

## 16-1. 영상 탐지의 기본 구조

영상 탐지의 기본 루프는 다음과 같습니다.

1. 영상에서 한 프레임을 읽습니다.
2. 프레임에 YOLO 추론을 적용합니다.
3. 결과 박스를 프레임 위에 그립니다.
4. 저장하거나 화면에 표시합니다.
5. 다음 프레임으로 넘어갑니다.

즉, 한 장 이미지 추론을 반복하는 구조입니다.

## 16-2. 예제 프레임 시퀀스 만들기

외부 영상 파일 없이도 실행할 수 있도록 움직이는 사각형 두 개로 프레임을 만듭니다. 실제 영상에서는 이 자리에 `cv2.VideoCapture`로 읽은 프레임이 들어갑니다.

In [ ]:
def make_frame(frame_idx, width=640, height=360):
    frame = np.full((height, width, 3), 245, dtype=np.uint8)

    cat_x = 50 + frame_idx * 18
    dog_x = 420 - frame_idx * 10

    frame[110:220, cat_x:cat_x + 130] = np.array([220, 80, 80], dtype=np.uint8)
    frame[140:255, dog_x:dog_x + 150] = np.array([70, 120, 220], dtype=np.uint8)
    return frame


frames = [make_frame(i) for i in range(8)]

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for idx, ax in enumerate(axes.flat):
    ax.imshow(frames[idx])
    ax.set_title(f'frame {idx}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 16-3. 프레임별 탐지 결과 형식 만들기

실제 모델이 없을 때도 영상 후처리 흐름을 연습할 수 있도록 프레임 번호에 따라 움직이는 예제 탐지 결과를 만듭니다.

In [ ]:
def fallback_video_detections(frame_idx):
    cat_x = 50 + frame_idx * 18
    dog_x = 420 - frame_idx * 10
    return [
        {'class': 'cat', 'score': 0.90 - frame_idx * 0.015, 'box': (cat_x, 110, cat_x + 130, 220)},
        {'class': 'dog', 'score': 0.84 + frame_idx * 0.010, 'box': (dog_x, 140, dog_x + 150, 255)},
    ]


for frame_idx in range(3):
    print('frame', frame_idx, fallback_video_detections(frame_idx))

## 16-4. 프레임 위에 탐지 결과 그리기

In [ ]:
def draw_frame_detections(ax, frame, detections, score_threshold=0.25):
    ax.imshow(frame)
    ax.axis('off')

    color_map = {'cat': 'crimson', 'dog': 'royalblue'}
    for det in detections:
        if det['score'] < score_threshold:
            continue
        x1, y1, x2, y2 = det['box']
        color = color_map.get(det['class'], 'seagreen')
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2.2))
        ax.text(
            x1,
            max(12, y1 - 5),
            f"{det['class']} {det['score']:.2f}",
            color='white',
            fontsize=9,
            weight='bold',
            bbox={'facecolor': color, 'alpha': 0.85, 'pad': 2, 'edgecolor': 'none'},
        )


fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for idx, ax in enumerate(axes.flat):
    detections = fallback_video_detections(idx)
    draw_frame_detections(ax, frames[idx], detections, score_threshold=0.25)
    ax.set_title(f'frame {idx}')
plt.tight_layout()
plt.show()

## 16-5. 실제 YOLO 모델을 사용할 때의 프레임 추론 함수

아래 함수는 `ultralytics` 모델이 준비되어 있을 때 한 프레임을 추론하고, 앞 노트북과 같은 탐지 결과 형식으로 변환합니다.

이 셀은 함수를 정의만 하므로 모델이 없어도 실행됩니다.

In [ ]:
def predict_frame_with_yolo(model, frame, conf=0.25):
    results = model.predict(source=frame, conf=conf, verbose=False)
    result = results[0]
    names = result.names
    detections = []

    for box in result.boxes:
        xyxy = box.xyxy[0].cpu().numpy().tolist()
        class_id = int(box.cls[0].cpu().item())
        score = float(box.conf[0].cpu().item())
        detections.append({
            'class': names[class_id],
            'score': score,
            'box': tuple(xyxy),
        })

    return detections

## 16-6. 실제 영상 파일 처리 코드

아래 코드는 실제 영상 파일을 읽고 결과 영상을 저장하는 기본 형태입니다. `video_path`, `output_path`, `model_path`를 본인 환경에 맞게 바꾼 뒤 사용합니다.

이 노트북에서는 기본값으로 실행하지 않도록 `run_real_video = False`로 둡니다.

In [ ]:
run_real_video = False
video_path = Path('input.mp4')
output_path = Path('output_yolo.mp4')
model_path = Path('yolov8n.pt')

if run_real_video:
    import cv2
    from ultralytics import YOLO

    model = YOLO(str(model_path))
    cap = cv2.VideoCapture(str(video_path))

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        detections = predict_frame_with_yolo(model, frame_rgb, conf=0.25)

        for det in detections:
            x1, y1, x2, y2 = [int(round(v)) for v in det['box']]
            cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (0, 0, 255), 2)
            cv2.putText(
                frame_bgr,
                f"{det['class']} {det['score']:.2f}",
                (x1, max(20, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 0, 255),
                2,
            )

        writer.write(frame_bgr)

    cap.release()
    writer.release()
    print('saved:', output_path)
else:
    print('실제 영상 처리는 비활성화되어 있습니다. run_real_video=True로 바꾸면 실행됩니다.')

## 16-7. Frame stride로 속도 조절하기

모든 프레임을 추론하면 가장 정확하지만 느릴 수 있습니다. 실습이나 빠른 확인에서는 몇 프레임마다 한 번씩 처리하는 `frame_stride`를 둘 수 있습니다.

In [ ]:
frame_stride = 2
processed = []

start = time.perf_counter()
for frame_idx, frame in enumerate(frames):
    if frame_idx % frame_stride != 0:
        continue
    detections = fallback_video_detections(frame_idx)
    processed.append((frame_idx, len(detections)))
elapsed = time.perf_counter() - start

print('frame_stride:', frame_stride)
print('processed frames:', processed)
print('elapsed seconds:', round(elapsed, 5))

## 16-8. 프레임별 탐지 개수 집계하기

영상에서는 한 프레임의 결과보다 시간에 따른 변화도 중요합니다. 예를 들어 사람 수, 차량 수, 특정 객체의 등장 여부를 프레임별로 기록할 수 있습니다.

In [ ]:
counts_by_frame = []

for frame_idx, frame in enumerate(frames):
    detections = fallback_video_detections(frame_idx)
    counts = {}
    for det in detections:
        if det['score'] < 0.5:
            continue
        counts[det['class']] = counts.get(det['class'], 0) + 1
    counts_by_frame.append({'frame': frame_idx, **counts})

for row in counts_by_frame:
    print(row)

## 16-9. 웹캠 추론 구조

웹캠도 영상 파일과 거의 같습니다. `cv2.VideoCapture(0)`으로 프레임을 읽고 같은 추론 함수를 적용합니다.

```python
import cv2
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
cap = cv2.VideoCapture(0)

while True:
    ok, frame_bgr = cap.read()
    if not ok:
        break

    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    detections = predict_frame_with_yolo(model, frame_rgb, conf=0.25)

    # frame_bgr 위에 bbox를 그리고 cv2.imshow로 표시
    if cv2.waitKey(1) == 27:
        break

cap.release()
cv2.destroyAllWindows()
```

노트북 환경에서는 GUI 창이 제한될 수 있으므로, 웹캠 실습은 `.py` 파일로 옮겨 실행하는 편이 안정적입니다.

## 정리

- 영상 탐지는 이미지 탐지를 프레임마다 반복하는 구조입니다.
- 실제 적용에서는 모델 정확도뿐 아니라 FPS, frame stride, 입출력 병목도 중요합니다.
- 탐지 결과를 프레임별로 저장하면 객체 수 변화나 이벤트 감지 같은 후속 분석을 할 수 있습니다.
- 웹캠, 영상 파일, 이미지 시퀀스는 입력 방식만 다르고 핵심 추론 함수는 거의 같습니다.

이제 2장의 흐름은 `분류와 탐지의 차이 -> bbox/IoU -> NMS/평가 -> 초기 탐지 방식 -> YOLO 아이디어 -> YOLO 출력 해석 -> 이미지 추론 -> 영상 추론`으로 이어집니다.